In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [2]:
df = pd.read_csv('../../data/raw/zomato.csv',encoding='latin1')
df.head(3)

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   str    
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   str    
 4   Address               9551 non-null   str    
 5   Locality              9551 non-null   str    
 6   Locality Verbose      9551 non-null   str    
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   str    
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   str    
 12  Has Table booking     9551 non-null   str    
 13  Has Online delivery   9551 non-null   str    
 14  Is delivering now     9551 non-null   str    
 15  Switch to order menu  9551 non-n

In [4]:
df.isnull().sum()

Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                9
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64

In [5]:
df.columns

Index(['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address',
       'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines',
       'Average Cost for two', 'Currency', 'Has Table booking',
       'Has Online delivery', 'Is delivering now', 'Switch to order menu',
       'Price range', 'Aggregate rating', 'Rating color', 'Rating text',
       'Votes'],
      dtype='str')

In [6]:
essential_cols = [
    'Restaurant ID', 'Restaurant Name', 'City',
    'Cuisines', 'Price range',
    'Aggregate rating', 'Has Table booking', 'Has Online delivery'
  ]
df2 = df[essential_cols].copy()
df2.info()
print(df2.columns)

<class 'pandas.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Restaurant ID        9551 non-null   int64  
 1   Restaurant Name      9551 non-null   str    
 2   City                 9551 non-null   str    
 3   Cuisines             9542 non-null   str    
 4   Price range          9551 non-null   int64  
 5   Aggregate rating     9551 non-null   float64
 6   Has Table booking    9551 non-null   str    
 7   Has Online delivery  9551 non-null   str    
dtypes: float64(1), int64(2), str(5)
memory usage: 1.0 MB
Index(['Restaurant ID', 'Restaurant Name', 'City', 'Cuisines', 'Price range',
       'Aggregate rating', 'Has Table booking', 'Has Online delivery'],
      dtype='str')


In [7]:
text_cols = [
    'Restaurant Name', 'City', 'Cuisines', 'Price range',
    'Has Table booking', 'Has Online delivery'
]
numeric_cols = [
    'Restaurant ID', 'Price range','Aggregate rating'
]

df2[text_cols] = df2[text_cols].fillna('')
df2[numeric_cols] = df2[numeric_cols].fillna(0)

df2 = df2[df2['Aggregate rating'] > 0]

df2.isnull().sum()

Restaurant ID          0
Restaurant Name        0
City                   0
Cuisines               0
Price range            0
Aggregate rating       0
Has Table booking      0
Has Online delivery    0
dtype: int64

In [8]:
def clean_tags(text):
    text = str(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r"[\[\]\"',]", '', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [9]:
df2['price_category'] = pd.cut(df2['Price range'], bins=4, labels=['very_cheap', 'cheap', 'medium', 'expensive'])
df2['rating_category'] = pd.cut(df2['Aggregate rating'], bins=4, labels=['poor', 'average', 'good', 'excellent'])


df2['booking_status'] = df2['Has Table booking'].map({'Yes': 'has_book', 'No': 'no_book'})
df2['delivery_status'] = df2['Has Online delivery'].map({'Yes': 'has_delivery', 'No': 'no_delivery'})

df2['combined_features'] = (
    (df2['Restaurant Name'] + ' ') +
    (df2['Cuisines'] + ' ') * 2 +
    df2['City'] + ' ' +
    df2['price_category'].astype(str) + ' ' +
    #df2['rating_category'].astype(str) + ' ' +
    df2['booking_status'] + ' ' +
    df2['delivery_status']
)

df2['combined_features'] = df2['combined_features'].apply(clean_tags)

restaurants = df2[[
    'Restaurant ID', 'combined_features', 'Restaurant Name',
    'City', 'Cuisines', 'Price range',
    'Aggregate rating', 'Has Table booking', 'Has Online delivery'
]]

restaurants = restaurants.reset_index(drop=True)

for name, features in zip(restaurants['Restaurant Name'].head(3), restaurants['combined_features'].head(3)):
    print(f"Name: {name}")
    print(f"Features: {features}\n")

Name: Le Petit Souffle
Features: le petit souffle french japanese desserts french japanese desserts makati city medium has_book no_delivery

Name: Izakaya Kikufuji
Features: izakaya kikufuji japanese japanese makati city medium has_book no_delivery

Name: Heat - Edsa Shangri-La
Features: heat - edsa shangri-la seafood asian filipino indian seafood asian filipino indian mandaluyong city expensive has_book no_delivery



In [10]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.

In [11]:
tfidf_matrix = tfidf.fit_transform(restaurants['combined_features'])
tfidf_matrix.shape

(7403, 5000)

In [12]:
indices = pd.Series(restaurants.index, index=restaurants['Restaurant Name']).drop_duplicates()

def get_recommendations(restaurant_name):
    if restaurant_name not in indices:
        return "Không tìm thấy nhà hàng này trong dữ liệu!"
    
    idx = indices[restaurant_name]
    query_vector = tfidf_matrix[idx]
    sim_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    related_indices = sim_scores.argsort()[-11:-1][::-1]  # Lấy top 10, bỏ chính nó

    top10 = restaurants.iloc[related_indices].copy()
    top10 = top10.sort_values('Aggregate rating', ascending=False)
    
    return top10[[
        'Restaurant Name', 'City', 'Cuisines', 
        'Price range', 'Aggregate rating',
        'Has Table booking', 'Has Online delivery'
    ]]

# Test 
recommendations = get_recommendations('Izakaya Kikufuji')
print(recommendations)

                   Restaurant Name         City                    Cuisines  \
0                 Le Petit Souffle  Makati City  French, Japanese, Desserts   
2679      Fuji Japanese Restaurant    New Delhi                    Japanese   
1324                       Kuuraku      Gurgaon                    Japanese   
472   Fuji Bay Japanese Restaurant   Sioux City             Japanese, Sushi   
1372                      Daikichi      Gurgaon                    Japanese   
3253                        Tamura    New Delhi                    Japanese   
5808                       Daitchi    New Delhi           Chinese, Japanese   
3641                        Tamura    New Delhi                    Japanese   
1392                       Komachi      Gurgaon                    Japanese   
1864                         Tokyo      Gurgaon                    Japanese   

      Price range  Aggregate rating Has Table booking Has Online delivery  
0               3               4.8               Yes 

In [14]:
def export_recommendations_to_file(restaurant_name, top_n=10, filename='recommendations.txt'):
    """
    Xuất kết quả gợi ý ra file text
    """
    # Lấy kết quả gợi ý
    if restaurant_name not in indices:
        return f"Không tìm thấy nhà hàng '{restaurant_name}' trong dữ liệu!"
    
    recommendations = get_recommendations(restaurant_name)
    
    # Lấy thông tin nhà hàng gốc
    idx = indices[restaurant_name]
    original_restaurant = restaurants.iloc[idx]
    
    # Ghi ra file
    with open(filename, 'w', encoding='utf-8') as f:
        
        # Thông tin nhà hàng đầu vào
        f.write("NHÀ HÀNG ĐẦU VÀO (INPUT):\n")
        f.write(f"Tên nhà hàng: {original_restaurant['Restaurant Name']}\n")
        f.write(f"Thành phố: {original_restaurant['City']}\n")
        f.write(f"Ẩm thực: {original_restaurant['Cuisines']}\n")
        f.write(f"Mức giá: {original_restaurant['Price range']}/4\n")
        f.write(f"Điểm đánh giá: {original_restaurant['Aggregate rating']}/5.0\n")
        f.write(f"Đặt bàn: {original_restaurant['Has Table booking']}\n")
        f.write(f"Giao hàng: {original_restaurant['Has Online delivery']}\n")
        f.write("\n\n")
        

        f.write("CÁC NHÀ HÀNG TƯƠNG TỰ (TOP {}):\n".format(len(recommendations)))        
        for i, (idx, row) in enumerate(recommendations.iterrows(), 1):
            f.write(f"{i}. {row['Restaurant Name']}\n")
            f.write(f"   - Thành phố: {row['City']}\n")
            f.write(f"   - Ẩm thực: {row['Cuisines']}\n")
            f.write(f"   - Mức giá: {row['Price range']}/4\n")
            f.write(f"   - Điểm đánh giá: {row['Aggregate rating']}/5.0\n")
            f.write(f"   - Đặt bàn: {row['Has Table booking']}\n")
            f.write(f"   - Giao hàng: {row['Has Online delivery']}\n")
            f.write("\n")
        
    
    print(f"Đã lưu kết quả vào file: {filename}")
    print(f"Đường dẫn: {os.path.abspath(filename)}")

# Chạy thử
import os
export_recommendations_to_file('Izakaya Kikufuji', top_n=10, filename='recommendation_results.txt')

Đã lưu kết quả vào file: recommendation_results.txt
Đường dẫn: d:\Restaraunt Rating\src\model_training\recommendation_results.txt
